# MaskRCNN + Focal Loss + Crop Lumbar
## Modelo J — MaskRCNN filtrado + preprocesamiento espinal + especialización L5

| Modelo | Arq | Dataset | Preproc | Loss | L5 |
|--------|-----|---------|---------|------|----|
| C | MaskRCNN | Filtrado | Curva+CLAHE | CE estándar | — |
| D | MaskRCNN | Completo | Curva+CLAHE | CE estándar | — |
| **J** | **MaskRCNN** | **Filtrado** | **Curva+CLAHE** | **Focal L5=3x** | **crop lumbar** |

### Diferencia clave vs C/D
- Loss personalizada: Focal Loss con peso 3x en L5, 2x en L1-L4
- Segunda etapa: crop lumbar L1-L5 para refinar predicciones
- Evaluación alineada con el pipeline MedSAM para comparación directa

### Pregunta de investigación
¿Puede MaskRCNN con especialización lumbar alcanzar resultados
comparables a MedSAM en L5, siendo una arquitectura más ligera?

## 0 — Instalación

In [ ]:
!pip install -q detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu118/torch2.0/index.html
!pip install -q scipy scikit-learn

import torch
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')
print(f'GPU:     {torch.cuda.get_device_name(0)}')
print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
import warnings, random, json, copy
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.model_selection import train_test_split
from scipy.interpolate import UnivariateSpline
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings('ignore')

# Detectron2
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.evaluation import COCOEvaluator
from detectron2.modeling import build_model
from detectron2.structures import Instances, Boxes
from detectron2.utils.logger import setup_logger
setup_logger()

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT   = Path('/content/drive/MyDrive')
DATASET_ROOT = DRIVE_ROOT / 'Scoliosis_Dataset'
CSV_PATH     = DATASET_ROOT / 'indice_dataset.csv'
METRICS_DIR  = DATASET_ROOT / 'RadiographMetrics'
CURVAS_DIR   = METRICS_DIR  / 'curvas_en_pixeles'
WORK_DIR     = Path('/content/maskrcnn_focal')
OUTPUT_J     = WORK_DIR / 'output_J'
OUTPUT_J_CROP= WORK_DIR / 'output_J_crop'
for d in [OUTPUT_J, OUTPUT_J_CROP]: d.mkdir(parents=True, exist_ok=True)

COL_SPLIT  = 'split'
COL_IMAGE  = 'radiograph_path'
COL_MASK   = 'multiclass_id_png'
COL_BINARY = 'label_binary_path'

CLASS_NAMES = ['T1','T2','T3','T4','T5','T6','T7','T8','T9','T10',
               'T11','T12','L1','L2','L3','L4','L5']
NUM_CLASSES = 17
LUMBAR_IDS  = list(range(12, 17))  # L1-L5
L5_ID       = 16
SEED        = 42
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f'✔ Configuración lista | Device: {DEVICE}')

---
## 1 — Datos, curvas y filtrado

In [ ]:
df = pd.read_csv(CSV_PATH, sep=';')
print(f'Total: {len(df)}')

CURVE_CACHE = {}
for _, row in df.iterrows():
    pid = int(row['patient_id'])
    p   = CURVAS_DIR / f'curva_pixeles_{pid}.csv'
    if p.exists():
        CURVE_CACHE[pid] = pd.read_csv(p)[['x_px','y_px']].values.astype(np.float32)
print(f'✔ Curvas: {len(CURVE_CACHE)}/{len(df)}')

def load_spine_curve(stem):
    m = df[df[COL_IMAGE].apply(lambda p: Path(p).stem == stem)]
    if len(m) == 0: return None
    return CURVE_CACHE.get(int(m.iloc[0]['patient_id']), None)

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df[COL_SPLIT], random_state=SEED)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df[COL_SPLIT], random_state=SEED)
for d in [train_df, val_df, test_df]: d.reset_index(drop=True, inplace=True)

def has_min_classes(mp, n=13):
    m = cv2.imread(str(mp), cv2.IMREAD_UNCHANGED)
    if m is None: return False
    if m.ndim == 3: m = m[:,:,0]
    return all(i in set(int(v) for v in np.unique(m) if v > 0)
               for i in range(1, n+1))

def filter_complete(df):
    return df[df[COL_MASK].apply(
        lambda p: has_min_classes(DATASET_ROOT/p)
    )].reset_index(drop=True)

print('Filtrando...')
train_f = filter_complete(train_df)
val_f   = filter_complete(val_df)
test_f  = filter_complete(test_df)
print(f'Filtrado: Train={len(train_f)} Val={len(val_f)} Test={len(test_f)}')

---
## 2 — Preprocesamiento espinal

In [ ]:
def build_spine_probability_map(curve_pts, h, w, sigma_x=45):
    prob_map = np.zeros((h,w), dtype=np.float32)
    pts = curve_pts[np.argsort(curve_pts[:,1])]
    ys  = pts[:,1].astype(float); xs = pts[:,0].astype(float)
    _, ui = np.unique(ys, return_index=True)
    ys=ys[ui]; xs=xs[ui]
    if len(ys) < 4: return prob_map
    try:
        sp = UnivariateSpline(ys, xs, k=min(3,len(ys)-1), s=len(ys)*10)
    except:
        sp = lambda y: np.interp(y, ys, xs)
    cx = np.clip(sp(np.arange(h,dtype=float)), 0, w-1)
    xg = np.arange(w, dtype=float)
    for ry in range(h):
        prob_map[ry] = np.exp(-0.5*((xg-cx[ry])/sigma_x)**2)
    return prob_map


def compute_crop_params(binary_path, img_h, img_w, margin=0.08):
    b = cv2.imread(str(binary_path), cv2.IMREAD_GRAYSCALE)
    if b is None: return None
    Hb,Wb = b.shape
    if (Hb>=Wb) != (img_h>=img_w):
        b=cv2.rotate(b,cv2.ROTATE_90_CLOCKWISE); Hb,Wb=b.shape
    if (Hb,Wb) != (img_h,img_w):
        b=cv2.resize(b,(img_w,img_h),interpolation=cv2.INTER_NEAREST)
    bb=(b>127).astype(np.uint8)
    rows=np.any(bb,axis=1); cols=np.any(bb,axis=0)
    if not rows.any(): return None
    y1,y2=np.where(rows)[0][[0,-1]]; x1,x2=np.where(cols)[0][[0,-1]]
    dy=max(1,int((y2-y1)*margin)); dx=max(1,int((x2-x1)*margin))
    return (max(0,x1-dx),max(0,y1-dy),min(img_w,x2+dx),min(img_h,y2+dy))


def preprocess_image(img_path, binary_path, curve_pts,
                     target_size=(512,1024), use_spine_map=True, alpha=0.3):
    img = cv2.imread(str(img_path))
    if img is None: return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    H,W  = gray.shape
    enhanced = cv2.createCLAHE(clipLimit=3.0,tileGridSize=(8,8)).apply(gray)
    if use_spine_map and curve_pts is not None and len(curve_pts)>=4:
        pm = build_spine_probability_map(curve_pts,H,W)
        enhanced = np.clip(
            enhanced.astype(np.float32)*(1+alpha*pm),0,255).astype(np.uint8)
    binary = cv2.imread(str(binary_path), cv2.IMREAD_GRAYSCALE)
    if binary is not None:
        Hb,Wb = binary.shape
        if (Hb>=Wb)!=(H>=W):
            binary=cv2.rotate(binary,cv2.ROTATE_90_CLOCKWISE); Hb,Wb=binary.shape
        if (Hb,Wb)!=(H,W):
            binary=cv2.resize(binary,(W,H),interpolation=cv2.INTER_NEAREST)
        bb=(binary>127).astype(np.uint8)
        rows=np.any(bb,axis=1); cols=np.any(bb,axis=0)
        if rows.any():
            y1,y2=np.where(rows)[0][[0,-1]]; x1,x2=np.where(cols)[0][[0,-1]]
            dy=max(1,int((y2-y1)*0.08)); dx=max(1,int((x2-x1)*0.08))
            enhanced=enhanced[max(0,y1-dy):min(H,y2+dy),
                              max(0,x1-dx):min(W,x2+dx)]
    tw,th = target_size
    hc,wc = enhanced.shape[:2]
    sc    = min(tw/wc, th/hc)
    nw,nh = int(wc*sc), int(hc*sc)
    res   = cv2.resize(enhanced,(nw,nh),interpolation=cv2.INTER_LINEAR)
    canvas= np.zeros((th,tw), dtype=np.uint8)
    py,px = (th-nh)//2, (tw-nw)//2
    canvas[py:py+nh,px:px+nw] = res
    return cv2.cvtColor(canvas, cv2.COLOR_GRAY2BGR)


def gt_from_png_aligned(mask_path, binary_path,
                         target_h=1024, target_w=512):
    m = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
    if m is None:
        return {c:np.zeros((target_h,target_w),np.uint8)
                for c in range(NUM_CLASSES)}
    if m.ndim==3: m=m[:,:,0]
    Ho,Wo = m.shape
    crop  = compute_crop_params(binary_path,Ho,Wo)
    if crop:
        x1,y1,x2,y2=crop; m=m[y1:y2,x1:x2]
    Hc,Wc = m.shape
    sc    = min(target_w/Wc, target_h/Hc)
    nw,nh = int(Wc*sc), int(Hc*sc)
    m     = cv2.resize(m,(nw,nh),interpolation=cv2.INTER_NEAREST)
    canvas= np.zeros((target_h,target_w), dtype=np.uint8)
    py,px = (target_h-nh)//2, (target_w-nw)//2
    canvas[py:py+nh,px:px+nw] = m
    return {c:(canvas==c+1).astype(np.uint8) for c in range(NUM_CLASSES)}


def mask_to_bbox(mask, margin=5):
    rows=np.any(mask,axis=1); cols=np.any(mask,axis=0)
    if not rows.any(): return None
    y1,y2=np.where(rows)[0][[0,-1]]; x1,x2=np.where(cols)[0][[0,-1]]
    H,W=mask.shape
    return [max(0,x1-margin),max(0,y1-margin),
            min(W,x2+margin),min(H,y2+margin)]


print('✔ Preprocesamiento definido')

---
## 3 — Registro COCO y dataset Detectron2

MaskRCNN en Detectron2 usa formato COCO JSON.
Se generan dos datasets:
- **dataset_J**: imágenes completas 512×1024, 17 clases
- **dataset_J_crop**: crops lumbares 512×512, 5 clases (L1-L5)

In [ ]:
def mask_to_rle(mask):
    """Convierte máscara binaria a RLE para COCO."""
    from pycocotools import mask as mask_utils
    rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    rle['counts'] = rle['counts'].decode('utf-8')
    return rle


def build_coco_dataset(split_df, img_dir, split_name,
                        lumbar_only=False, target_size=(512,1024)):
    """
    Construye dataset COCO JSON para Detectron2.
    lumbar_only=True → solo L1-L5, reclasificadas como 0-4
    """
    img_dir = Path(img_dir)
    img_dir.mkdir(parents=True, exist_ok=True)
    tw, th  = target_size
    classes = LUMBAR_IDS if lumbar_only else list(range(NUM_CLASSES))
    records = []
    ann_id  = 0

    for img_id, (_, row) in enumerate(split_df.iterrows()):
        ip   = DATASET_ROOT / row[COL_IMAGE]
        mp   = DATASET_ROOT / row[COL_MASK]
        bp   = DATASET_ROOT / row[COL_BINARY]
        stem = ip.stem
        curve= load_spine_curve(stem)

        proc = preprocess_image(ip, bp, curve,
                                 target_size=target_size,
                                 use_spine_map=True)
        if proc is None: continue

        # Guardar imagen preprocesada
        img_path = img_dir / f'{stem}.jpg'
        cv2.imwrite(str(img_path), proc,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])

        # GT alineado
        gt = gt_from_png_aligned(mp, bp, th, tw)

        annotations = []
        for c in classes:
            if gt[c].sum() == 0: continue
            bbox = mask_to_bbox(gt[c], margin=2)
            if bbox is None: continue
            x1,y1,x2,y2 = bbox
            # Clase remapeada: lumbar_only → 0-4, completo → 0-16
            cat_id = LUMBAR_IDS.index(c) if lumbar_only else c
            annotations.append({
                'id'          : ann_id,
                'image_id'    : img_id,
                'category_id' : cat_id,
                'bbox'        : [x1, y1, x2-x1, y2-y1],
                'area'        : int(gt[c].sum()),
                'segmentation': mask_to_rle(gt[c]),
                'iscrowd'     : 0
            })
            ann_id += 1

        if not annotations: continue
        records.append({
            'id'          : img_id,
            'file_name'   : str(img_path),
            'height'      : th,
            'width'       : tw,
            'annotations' : annotations
        })

    # Categorías
    if lumbar_only:
        cats = [{'id':i,'name':CLASS_NAMES[c]}
                for i,c in enumerate(LUMBAR_IDS)]
    else:
        cats = [{'id':i,'name':CLASS_NAMES[i]}
                for i in range(NUM_CLASSES)]

    coco = {
        'images'     : [{'id':r['id'],'file_name':r['file_name'],
                         'height':r['height'],'width':r['width']}
                        for r in records],
        'annotations': [a for r in records for a in r['annotations']],
        'categories' : cats
    }

    json_path = img_dir.parent / f'annotations_{split_name}.json'
    with open(json_path,'w') as f: json.dump(coco,f)
    print(f'  {split_name}: {len(records)} imgs, '
          f'{len(coco["annotations"])} anotaciones → {json_path}')
    return json_path


# ── Dataset J: imagen completa 17 clases ─────────────────────
DS_J    = WORK_DIR / 'dataset_J'
IMG_J   = DS_J / 'images'
print('Construyendo Dataset J (17 clases, 512×1024)...')
for split,df_s in [('train',train_f),('val',val_f),('test',test_f)]:
    build_coco_dataset(df_s, IMG_J/split, split,
                        lumbar_only=False, target_size=(512,1024))

# ── Dataset J_crop: crop lumbar 5 clases ─────────────────────
DS_CROP  = WORK_DIR / 'dataset_J_crop'
IMG_CROP = DS_CROP / 'images'
print('\nConstruyendo Dataset J_crop (L1-L5, 512×512)...')
for split,df_s in [('train',train_f),('val',val_f),('test',test_f)]:
    build_coco_dataset(df_s, IMG_CROP/split, split,
                        lumbar_only=True, target_size=(512,512))

print('\n✔ Datasets listos')

In [ ]:
# Registrar datasets en Detectron2
for split in ['train','val','test']:
    # Dataset completo
    name = f'spine_J_{split}'
    if name in DatasetCatalog:
        DatasetCatalog.remove(name)
        MetadataCatalog.remove(name)
    register_coco_instances(
        name,
        {},
        str(DS_J / f'annotations_{split}.json'),
        str(IMG_J / split)
    )
    MetadataCatalog.get(name).thing_classes = CLASS_NAMES

    # Dataset lumbar
    name_crop = f'spine_J_crop_{split}'
    if name_crop in DatasetCatalog:
        DatasetCatalog.remove(name_crop)
        MetadataCatalog.remove(name_crop)
    register_coco_instances(
        name_crop,
        {},
        str(DS_CROP / f'annotations_{split}.json'),
        str(IMG_CROP / split)
    )
    MetadataCatalog.get(name_crop).thing_classes = [
        CLASS_NAMES[c] for c in LUMBAR_IDS]

print('✔ Datasets registrados en Detectron2')

---
## 4 — Focal Loss para MaskRCNN

Detectron2 no tiene Focal Loss de clasificación de instancias nativa.
Se extiende el trainer para reemplazar la CE estándar por una
versión ponderada por clase, con L5=3x y L1-L4=2x.

In [ ]:
from detectron2.modeling.roi_heads.fast_rcnn import FastRCNNOutputLayers
from detectron2.modeling.roi_heads import StandardROIHeads
from detectron2.utils.registry import Registry

# Pesos por clase para Focal Loss
# índice 0-16 = T1-L5, índice 17 = background
def get_class_weights(num_classes, lumbar_only=False, device='cuda'):
    """
    Retorna tensor de pesos por clase.
    L5 = índice 16 (completo) o índice 4 (lumbar_only)
    """
    weights = torch.ones(num_classes + 1)  # +1 para background
    if lumbar_only:
        # L1=0, L2=1, L3=2, L4=3, L5=4
        weights[0] = 2.0  # L1
        weights[1] = 2.0  # L2
        weights[2] = 2.0  # L3
        weights[3] = 2.0  # L4
        weights[4] = 3.0  # L5
    else:
        # T1-T12 = 0-11, L1-L5 = 12-16
        for i in range(6, 11):  # T7-T11
            weights[i] = 1.5
        for i in range(12, 16): # L1-L4
            weights[i] = 2.0
        weights[16] = 3.0       # L5
    return weights.to(device)


class FocalLossMaskRCNN(nn.Module):
    """
    Focal Loss con pesos de clase para MaskRCNN.
    Reemplaza la CrossEntropy estándar en la cabeza de clasificación.
    """
    def __init__(self, gamma=2.0, class_weights=None):
        super().__init__()
        self.gamma   = gamma
        self.weights = class_weights  # tensor [num_classes+1]

    def forward(self, logits, targets):
        """
        logits:  [N, num_classes+1]
        targets: [N] con índices de clase
        """
        ce   = F.cross_entropy(logits, targets, reduction='none',
                               weight=self.weights)
        pt   = torch.exp(-ce)
        focal= ((1-pt)**self.gamma * ce)
        return focal.mean()


class SpineTrainerFocal(DefaultTrainer):
    """
    Trainer MaskRCNN con Focal Loss ponderada.
    Sobreescribe la loss de clasificación de ROI heads.
    """
    def __init__(self, cfg, lumbar_only=False):
        super().__init__(cfg)
        self.lumbar_only = lumbar_only

    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = cfg.OUTPUT_DIR
        return COCOEvaluator(dataset_name,
                             output_dir=output_folder)

    def build_model(self, cfg):
        model = build_model(cfg)
        # Inyectar Focal Loss en la cabeza de clasificación
        nc    = cfg.MODEL.ROI_HEADS.NUM_CLASSES
        cw    = get_class_weights(nc,
                                   lumbar_only=self.lumbar_only,
                                   device=cfg.MODEL.DEVICE)
        focal = FocalLossMaskRCNN(gamma=2.0, class_weights=cw)

        # Patch: reemplazar box_predictor loss
        orig_forward = model.roi_heads.box_predictor.forward

        def patched_losses(predictions, proposals):
            scores, proposal_deltas = predictions
            gt_classes = torch.cat(
                [p.gt_classes for p in proposals], dim=0)
            # Focal Loss en lugar de CE
            cls_loss = focal(scores, gt_classes)
            # Box regression loss: mantener original
            from detectron2.modeling.box_regression import Box2BoxTransform
            gt_boxes = torch.cat(
                [p.gt_boxes.tensor for p in proposals], dim=0)
            box_loss = F.smooth_l1_loss(
                proposal_deltas,
                model.roi_heads.box_predictor.box2box_transform.get_deltas(
                    torch.cat([p.proposal_boxes.tensor for p in proposals]),
                    gt_boxes
                ),
                reduction='mean'
            )
            return {'loss_cls': cls_loss, 'loss_box_reg': box_loss}

        model.roi_heads.box_predictor.losses = patched_losses
        return model


print('✔ Focal Loss y SpineTrainerFocal definidos')

---
## 5 — Configuración MaskRCNN

In [ ]:
def get_maskrcnn_cfg(dataset_train, dataset_val, output_dir,
                      num_classes, max_iter=6000,
                      img_h=1024, img_w=512):
    cfg = get_cfg()
    cfg.merge_from_file(
        model_zoo.get_config_file(
            'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'))

    cfg.DATASETS.TRAIN = (dataset_train,)
    cfg.DATASETS.TEST  = (dataset_val,)
    cfg.OUTPUT_DIR     = str(output_dir)

    # Pesos preentrenados COCO
    cfg.MODEL.WEIGHTS  = model_zoo.get_checkpoint_url(
        'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml')

    cfg.MODEL.ROI_HEADS.NUM_CLASSES        = num_classes
    cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 256
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST  = 0.25

    # Tamaño de imagen
    cfg.INPUT.MIN_SIZE_TRAIN = (img_h,)
    cfg.INPUT.MAX_SIZE_TRAIN = max(img_h, img_w)
    cfg.INPUT.MIN_SIZE_TEST  = img_h
    cfg.INPUT.MAX_SIZE_TEST  = max(img_h, img_w)

    # Entrenamiento
    cfg.SOLVER.IMS_PER_BATCH = 4
    cfg.SOLVER.BASE_LR       = 0.0005
    cfg.SOLVER.MAX_ITER      = max_iter
    cfg.SOLVER.WARMUP_ITERS  = 200
    cfg.SOLVER.STEPS         = (int(max_iter*0.7), int(max_iter*0.9))
    cfg.SOLVER.GAMMA         = 0.1
    cfg.SOLVER.CHECKPOINT_PERIOD = 1000

    cfg.TEST.EVAL_PERIOD = 1000
    cfg.DATALOADER.NUM_WORKERS = 4
    cfg.MODEL.DEVICE = 'cuda'
    cfg.SEED = SEED

    Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg


# Modelo J — imagen completa 17 clases
cfg_j = get_maskrcnn_cfg(
    dataset_train = 'spine_J_train',
    dataset_val   = 'spine_J_val',
    output_dir    = OUTPUT_J,
    num_classes   = NUM_CLASSES,
    max_iter      = 6000,
    img_h=1024, img_w=512
)

# Modelo J_crop — crop lumbar 5 clases
cfg_j_crop = get_maskrcnn_cfg(
    dataset_train = 'spine_J_crop_train',
    dataset_val   = 'spine_J_crop_val',
    output_dir    = OUTPUT_J_CROP,
    num_classes   = 5,  # L1-L5
    max_iter      = 4000,
    img_h=512, img_w=512
)

print('✔ Configuraciones listas')
print(f'  J global: {cfg_j.SOLVER.MAX_ITER} iters | '
      f'{NUM_CLASSES} clases | 512×1024')
print(f'  J crop:   {cfg_j_crop.SOLVER.MAX_ITER} iters | '
      f'5 clases (L1-L5) | 512×512')

---
## 6 — Entrenamiento Modelo J (imagen completa)

In [ ]:
print('='*60)
print('MODELO J — MaskRCNN + Focal Loss (17 clases, 512×1024)')
print('='*60)

trainer_j = SpineTrainerFocal(cfg_j, lumbar_only=False)
trainer_j.resume_or_load(resume=False)
trainer_j.train()

print('\n✔ Modelo J entrenado')

---
## 7 — Entrenamiento Modelo J_crop (región lumbar)

In [ ]:
print('='*60)
print('MODELO J_CROP — MaskRCNN + Focal Loss (L1-L5, 512×512)')
print('='*60)

trainer_j_crop = SpineTrainerFocal(cfg_j_crop, lumbar_only=True)
trainer_j_crop.resume_or_load(resume=False)
trainer_j_crop.train()

print('\n✔ Modelo J_crop entrenado')

---
## 8 — Evaluación Dice por clase

In [ ]:
# Cargar predictores
cfg_j.MODEL.WEIGHTS = str(OUTPUT_J / 'model_final.pth')
cfg_j.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.25
predictor_j = DefaultPredictor(cfg_j)

cfg_j_crop.MODEL.WEIGHTS = str(OUTPUT_J_CROP / 'model_final.pth')
cfg_j_crop.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.25
predictor_j_crop = DefaultPredictor(cfg_j_crop)

print('✔ Predictores cargados')


def dice_iou(p, g):
    p,g = p.astype(bool), g.astype(bool)
    inter=(p&g).sum(); union=(p|g).sum()
    return (2*inter/(p.sum()+g.sum()) if (p.sum()+g.sum())>0 else 1.0,
            inter/union if union>0 else 1.0)


def maskrcnn_pred_to_masks(result, num_classes, H, W):
    """Convierte predicción Detectron2 a dict de máscaras binarias."""
    pred = {c:np.zeros((H,W),np.uint8) for c in range(num_classes)}
    inst = result['instances'].to('cpu')
    if len(inst) == 0: return pred
    ca = inst.pred_classes.numpy()
    ma = inst.pred_masks.numpy()
    sc = inst.scores.numpy()
    for c in range(num_classes):
        idx = np.where(ca==c)[0]
        if len(idx)==0: continue
        best = idx[np.argmax(sc[idx])]
        pred[c] = ma[best].astype(np.uint8)
    return pred


def extract_lumbar_crop_from_pred(proc_img, pred_masks_global,
                                   margin=40, target=512):
    """
    Extrae crop lumbar usando predicciones globales de J
    (en inferencia no hay GT disponible).
    """
    union = np.zeros(proc_img.shape[:2], np.uint8)
    for c in LUMBAR_IDS:
        union = np.maximum(union, pred_masks_global[c])

    # Fallback: usar GT si la predicción está vacía
    if union.sum() == 0: return None, None

    rows=np.where(np.any(union,axis=1))[0]
    cols=np.where(np.any(union,axis=0))[0]
    y1=max(0,rows.min()-margin); y2=min(proc_img.shape[0],rows.max()+margin)
    x1=max(0,cols.min()-margin); x2=min(proc_img.shape[1],cols.max()+margin)
    crop   = proc_img[y1:y2,x1:x2]
    crop_r = cv2.resize(crop,(target,target),interpolation=cv2.INTER_LINEAR)
    return crop_r, (x1,y1,x2,y2)


def eval_model_J(predictor_global, predictor_crop,
                  split_df, model_name):
    """
    Evaluación con fusión:
    T1-T12: predictor_global
    L1-L5:  predictor_crop (mayor resolución efectiva)
    """
    dice_cls = {c:[] for c in range(NUM_CLASSES)}
    iou_cls  = {c:[] for c in range(NUM_CLASSES)}
    l5_log   = []

    for _,row in split_df.iterrows():
        ip   = DATASET_ROOT/row[COL_IMAGE]
        mp   = DATASET_ROOT/row[COL_MASK]
        bp   = DATASET_ROOT/row[COL_BINARY]
        stem = ip.stem
        curve= load_spine_curve(stem)

        # Preprocesar imagen completa
        proc = preprocess_image(ip,bp,curve,
                                 target_size=(512,1024),use_spine_map=True)
        if proc is None: continue
        H,W  = proc.shape[:2]

        # GT alineado
        gt = gt_from_png_aligned(mp,bp,H,W)

        # Predicción global (T1-T12 + contexto lumbar)
        result_global = predictor_global(proc)
        pred_global   = maskrcnn_pred_to_masks(result_global,NUM_CLASSES,H,W)

        # Extraer crop lumbar
        crop,coords = extract_lumbar_crop_from_pred(
            proc, pred_global, margin=40)

        # Predicción crop lumbar
        pred_crop = None
        if crop is not None:
            result_crop = predictor_crop(crop)
            pred_crop   = maskrcnn_pred_to_masks(result_crop,5,512,512)

        # Evaluación por clase
        for c in range(NUM_CLASSES):
            if gt[c].sum()==0: continue

            if c in LUMBAR_IDS and pred_crop is not None and coords is not None:
                # Usar predicción del crop re-proyectada
                lumbar_idx = LUMBAR_IDS.index(c)
                x1c,y1c,x2c,y2c = coords
                m512 = pred_crop[lumbar_idx]
                canvas = np.zeros((H,W),np.uint8)
                m_reproj = cv2.resize(m512,(x2c-x1c,y2c-y1c),
                                      interpolation=cv2.INTER_NEAREST)
                canvas[y1c:y2c,x1c:x2c] = m_reproj
                pred_mask = canvas
            else:
                pred_mask = pred_global[c]

            d,iou = dice_iou(pred_mask,gt[c])
            dice_cls[c].append(d); iou_cls[c].append(iou)

            if c==L5_ID:
                l5_log.append({
                    'image'   :ip.name,'model':model_name,
                    'split'   :'Scoliosis' if stem.startswith('S_') else 'Normal',
                    'dice':d,'iou':iou,
                    'gt_px'   :int(gt[c].sum()),'pred_px':int(pred_mask.sum()),
                    'detected':pred_mask.sum()>0
                })

    return dice_cls, iou_cls, l5_log


print('Evaluando Modelo J (fusión global + crop lumbar)...')
dice_j, iou_j, l5_j = eval_model_J(
    predictor_j, predictor_j_crop, test_f, 'J_MaskRCNN_focal_crop')
print('✔ Evaluación completa')

---
## 9 — Tabla comparativa final

In [ ]:
# Cargar resultados previos
dice_prev = {
    # C: MaskRCNN filtrado sin focal (AP50=36.8%, Dice estimado)
    'C': dict(zip(range(17),[0.1640,0.1692,0.1447,0.1316,0.1723,0.1483,
                              0.1072,0.0910,0.1440,0.1461,0.1412,0.1262,
                              0.0862,0.0876,0.1375,0.1289,0.0584])),
    # G: MedSAM base
    'G': dict(zip(range(17),[0.8883,0.8914,0.8877,0.8955,0.8931,0.8917,
                              0.9067,0.8987,0.8872,0.8931,0.8908,0.9096,
                              0.9092,0.9036,0.9119,0.9104,0.9058])),
    # I: MedSAM + Focal + crop
    'I': dict(zip(range(17),[0.8807,0.8874,0.8768,0.8923,0.8927,0.8959,
                              0.9042,0.8919,0.8870,0.8911,0.8897,0.9082,
                              0.9036,0.9008,0.8959,0.9118,0.9323]))
}

print(f'\n{"Clase":<6} {"C (orig)":>9} {"J (focal)":>10} '
      f'{"G (MedSAM)":>11} {"I (Med+crop)":>13} {"Mejor":>7}')
print('─'*65)

rows = []
for c in range(NUM_CLASSES):
    dc = dice_prev['C'].get(c,0.)
    dj = np.mean(dice_j[c]) if dice_j[c] else 0.
    dg = dice_prev['G'].get(c,0.)
    di = dice_prev['I'].get(c,0.)
    bl = ['C','J','G','I'][[dc,dj,dg,di].index(max(dc,dj,dg,di))]
    tag= ' ◄L5' if c==L5_ID else ''
    print(f'{CLASS_NAMES[c]:<6} {dc:>9.4f} {dj:>10.4f} '
          f'{dg:>11.4f} {di:>13.4f} {bl:>7}{tag}')
    rows.append({'clase':CLASS_NAMES[c],'class_id':c,
                 'dice_C':dc,'dice_J':dj,'dice_G':dg,'dice_I':di,
                 'mejor':bl,
                 'iou_J':np.mean(iou_j[c]) if iou_j[c] else 0.})

print('─'*65)
mc = np.mean([r['dice_C'] for r in rows])
mj = np.mean([r['dice_J'] for r in rows])
mg = np.mean([r['dice_G'] for r in rows])
mi = np.mean([r['dice_I'] for r in rows])
print(f'{"MEAN":<6} {mc:>9.4f} {mj:>10.4f} {mg:>11.4f} {mi:>13.4f}')
print(f'\n  C  MaskRCNN filtrado (original)  : {mc:.4f}')
print(f'  J  MaskRCNN + Focal + crop       : {mj:.4f}  Δ vs C: {mj-mc:+.4f}')
print(f'  G  MedSAM base                   : {mg:.4f}  Δ vs J: {mg-mj:+.4f}')
print(f'  I  MedSAM + Focal + crop         : {mi:.4f}  Δ vs J: {mi-mj:+.4f}')
print(f'  Paper anterior                   : 0.7400')
print(f'\n  L5:')
l5_j_dice = np.mean(dice_j[L5_ID]) if dice_j[L5_ID] else 0.
print(f'  C={dice_prev["C"].get(16,0.):.4f} | '
      f'J={l5_j_dice:.4f} | '
      f'G={dice_prev["G"].get(16,0.):.4f} | '
      f'I={dice_prev["I"].get(16,0.):.4f}')

pd.DataFrame(rows).to_csv(
    DRIVE_ROOT/'models'/'comparacion_J.csv', index=False)
print('\n✔ Tabla guardada')

In [ ]:
# ── Figura comparativa estilo paper ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22,7))
fig.patch.set_facecolor('white')
x=np.arange(NUM_CLASSES); w=0.20
colors={'C':'#888888','J':'#E24B4A','G':'#0F6E56','I':'#185FA5'}

# Panel (a) — Dice por clase
for off,(lbl,col,vals) in enumerate([
    ('C orig',  colors['C'], [r['dice_C'] for r in rows]),
    ('J focal', colors['J'], [r['dice_J'] for r in rows]),
    ('G MedSAM',colors['G'], [r['dice_G'] for r in rows]),
    ('I Med+crop',colors['I'],[r['dice_I'] for r in rows]),
]):
    axes[0].bar(x+(off-1.5)*w,vals,w,label=lbl,
                color=col,alpha=0.85,edgecolor='k',lw=0.4)
axes[0].axhline(0.74,color='gray',ls=':',lw=1.5,label='Paper (0.74)')
axes[0].set_xticks(x); axes[0].set_xticklabels(CLASS_NAMES,rotation=45,fontsize=8)
axes[0].set_ylabel('Dice',fontsize=11); axes[0].set_ylim(0,1.05)
axes[0].set_title('(a) Dice por clase vertebral',fontsize=11,fontweight='bold')
axes[0].legend(fontsize=8,ncol=2); axes[0].grid(axis='y',alpha=0.3)

# Panel (b) — Delta C→J (impacto Focal Loss en MaskRCNN)
delta_cj = [r['dice_J']-r['dice_C'] for r in rows]
axes[1].bar(x, delta_cj,
            color=[colors['J'] if d>=0 else '#888888' for d in delta_cj],
            edgecolor='k',lw=0.4,alpha=0.85)
axes[1].axhline(0,color='black',lw=1.5)
axes[1].axhline(np.mean(delta_cj),color=colors['J'],ls='--',lw=1.5,
                label=f'Media={np.mean(delta_cj):+.3f}')
axes[1].set_xticks(x); axes[1].set_xticklabels(CLASS_NAMES,rotation=45,fontsize=8)
axes[1].set_ylabel('Δ Dice (J − C)',fontsize=11)
axes[1].set_title('(b) Impacto Focal Loss + crop\n(MaskRCNN: J vs C)',
                  fontsize=11,fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(axis='y',alpha=0.3)

# Panel (c) — Comparación L5 por arquitectura
arch_names = ['C\nMaskRCNN', 'J\nMaskRCNN+F', 'G\nMedSAM', 'I\nMedSAM+F']
l5_means   = [dice_prev['C'].get(16,0.), l5_j_dice,
               dice_prev['G'].get(16,0.), dice_prev['I'].get(16,0.)]
bar_colors = [colors['C'],colors['J'],colors['G'],colors['I']]
bars = axes[2].bar(arch_names, l5_means, color=bar_colors,
                    edgecolor='k',lw=0.8,alpha=0.85,width=0.5)
for bar,v in zip(bars,l5_means):
    axes[2].text(bar.get_x()+bar.get_width()/2, v+0.005,
                 f'{v:.3f}',ha='center',fontsize=9,fontweight='bold')
axes[2].axhline(0.52,color='gray',ls=':',lw=1.5,label='Paper L5=0.52')
axes[2].axhline(0.74,color='gray',ls='--',lw=1.5,label='Paper global=0.74')
axes[2].set_ylabel('Dice L5',fontsize=11); axes[2].set_ylim(0,1.05)
axes[2].set_title('(c) L5 por arquitectura\nMaskRCNN vs MedSAM',
                  fontsize=11,fontweight='bold')
axes[2].legend(fontsize=8); axes[2].grid(axis='y',alpha=0.3)

plt.suptitle(
    'MaskRCNN con Focal Loss y Crop Lumbar vs MedSAM — Comparación de Arquitecturas',
    fontsize=12,fontweight='bold',y=1.01)
plt.tight_layout()
plt.savefig(str(DRIVE_ROOT/'models'/'comparacion_J_vs_MedSAM.png'),
            dpi=200,bbox_inches='tight',facecolor='white')
plt.show()
print('✔ Figura guardada')

In [ ]:
# Análisis L5 Normal vs Escoliosis
if l5_j:
    l5_df = pd.DataFrame(l5_j)
    print('═══ L5 — Modelo J ═══════════════════════')
    print(l5_df.groupby('split')[['dice','detected']].agg(
        {'dice':['mean','median','std'],'detected':'mean'}
    ).round(4).to_string())
    l5_df.to_csv(DRIVE_ROOT/'models'/'l5_J.csv', index=False)
    print('✔ L5 guardado')

# Guardar modelo
import shutil
save_dir = DRIVE_ROOT/'models'/'maskrcnn_J'
save_dir.mkdir(exist_ok=True)
for src,dst in [
    (OUTPUT_J/'model_final.pth',      save_dir/'J_global_final.pth'),
    (OUTPUT_J_CROP/'model_final.pth', save_dir/'J_crop_final.pth'),
]:
    if src.exists(): shutil.copy(src,dst); print(f'✔ {dst.name} guardado')

print('\n'+'='*65)
print('RESUMEN FINAL — Modelo J (MaskRCNN + Focal + crop lumbar)')
print('='*65)
print(f'''
C  MaskRCNN original (filtrado)  : {mc:.4f}
J  MaskRCNN + Focal + crop       : {mj:.4f}  Δ vs C: {mj-mc:+.4f}
G  MedSAM base                   : {mg:.4f}  Δ vs J: {mg-mj:+.4f}
I  MedSAM + Focal + crop         : {mi:.4f}
Paper anterior                   : 0.7400

L5:
  C={dice_prev["C"].get(16,0.):.4f}  J={l5_j_dice:.4f}  
  G={dice_prev["G"].get(16,0.):.4f}  I={dice_prev["I"].get(16,0.):.4f}
''')
print('='*65)